# Transfer test: does the warning sign hold on other printers and in 2026?
**Internet of Production – AI, week 3, block 5** · about 30 min · work in pairs

**Thread:** In weeks 1 and 2 you found the warning sign (the energy per minute rises before an extruder blockage) and checked it on **one** printer in **2024/25**. The monitor of block 6 will run on **other printers** and on **new data**. Before we build it we ask: does the model transfer?

**Set-up**
- **Training (frozen):** KNN (k = 25) and the z-value rule, both fitted on the first 70 % of printer 1 in 2024/25 (as in week 1 and exercise 2b). They are **not** retrained.
- **Test:** the six printers of 2026 (5 Jan to 19 Sep 2026). The farm has been extended by three printers.
- **Question:** does a frozen model beat a limit that is set anew for every printer (*per-printer baseline*)?

**Write down your expectation before you run anything:** what do you expect for AUC, recall and false alarms on the new printers?

**Your expectation (one or two sentences):**

*Your answer:*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, recall_score, precision_score

DATA_OLD = Path('../../00_Data/printfarm')          # 2024/25, printers 0-2: used for TRAINING (as in weeks 1 and 2)
DATA_NEW = Path('../../00_Data/printfarm_2026')     # 2026, printers 0-5: used for TESTING
HORIZON = 360                                       # minutes
FEATURES = ['E30', 'E120', 'slope']

def make_blockage_data(folder, k, ts_format=None, horizon=HORIZON):
    """The pipeline of exercise 2 in one function: one row per 10th printing minute. ts_format=None: ISO time stamps (2026)."""
    d = pd.read_csv(folder / f'Printer_{k}.csv', sep=';', decimal=',', low_memory=False).dropna(how='all')
    d['Timestamp'] = pd.to_datetime(d['Timestamp'], format=ts_format)
    d = d.set_index('Timestamp')
    d['Energy'] = d['Total_Power'].diff()
    e = pd.read_csv(folder / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format=ts_format)
    blockages = pd.to_datetime(np.sort(e.loc[e['Error'] == 'Extruder blocked', 'Timestamp'].values))
    p = d[d['State'] == 'printing'].copy()
    p['E'] = p['Energy'] * 1000                              # Wh per minute
    idx = p.index.values
    nxt = np.searchsorted(blockages.values, idx)
    mtb = np.where(nxt < len(blockages), (blockages.values[np.minimum(nxt, len(blockages) - 1)] - idx) / np.timedelta64(1, 'm'), np.inf)
    p['y'] = ((mtb > 0) & (mtb <= horizon)).astype(int)      # only used to EVALUATE
    p['E30'] = p['E'].rolling(30).mean()
    p['E120'] = p['E'].rolling(120).mean()
    p['slope'] = p['E120'] - p['E120'].shift(240)
    return p.dropna(subset=FEATURES).iloc[::10], blockages

def evaluate_alarms(alarm, test, blockages):
    """alarm: boolean Series on the index of `test`. Blockages announced within 6 h and false alarm episodes per week."""
    t0, t1 = test.index[0], test.index[-1]
    blk = [t for t in blockages if t0 <= t <= t1]
    announced = sum(bool(alarm[(alarm.index >= t - pd.Timedelta(hours=6)) & (alarm.index <= t)].any()) for t in blk)
    times = alarm[alarm].index
    weeks = (t1 - t0).days / 7
    if len(times) == 0:
        return announced, len(blk), 0.0
    new_episode = np.concatenate([[0], np.diff(times.values) > np.timedelta64(6, 'h')])   # a gap of more than 6 h starts a new episode
    ep = pd.DataFrame({'ep': np.cumsum(new_episode), 'y': test.loc[times, 'y'].values}).groupby('ep')['y'].max()
    return announced, len(blk), float((ep == 0).sum() / weeks)

def wilson(k, n, z=1.96):
    """95 % interval of a share k / n (Wilson)."""
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half

def score_row(printer, model, test, blockages, score, alarm):
    ann, n_blk, fa = evaluate_alarms(alarm, test, blockages)
    return {'printer': printer, 'model': model, 'AUC': roc_auc_score(test['y'], score), 'recall': recall_score(test['y'], alarm),
            'precision': precision_score(test['y'], alarm, zero_division=0), 'announced': ann, 'blockages': n_blk, 'false episodes / week': fa}

## Step 1: Freeze the reference models
Fit KNN and the z-value limit on the first 70 % of printer 1 from 2024/25. From here on they are not changed.

In [ ]:
old, old_blockages = make_blockage_data(DATA_OLD, 1, ts_format='%d.%m.%Y %H:%M')
reference_old = old.iloc[:int(len(old) * ___)]

knn_frozen = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)).fit(reference_old[FEATURES], reference_old['y'])
mu_old, sd_old = reference_old['E120'].mean(), reference_old['E120'].std()
z_limit_old = np.quantile((reference_old['E120'] - mu_old) / sd_old, ___)
print(f'normal E120 (2024/25, printer 1): {mu_old:.3f} +/- {sd_old:.3f} Wh/min; limit at z = {z_limit_old:.2f}, i.e. E120 > {mu_old + z_limit_old * sd_old:.3f}')

## Step 2: Apply the frozen models to the six printers of 2026
Whole year, no retraining. For every printer: AUC, recall, blockages announced within 6 h and false alarm episodes per week. The share of announced blockages gets a 95 % interval (Wilson) because it rests on only 40 to 60 events.

In [ ]:
printers = {k: make_blockage_data(DATA_NEW, k) for k in range(6)}

rows = []
for k, (d, blk) in printers.items():
    z = (d['E120'] - mu_old) / sd_old
    rows.append(score_row(k, 'KNN (frozen)', d, blk, knn_frozen.predict_proba(d[FEATURES])[:, 1], pd.Series(knn_frozen.predict(d[FEATURES]) == 1, index=d.index)))
    rows.append(score_row(k, 'z-value rule (frozen)', d, blk, z, z > z_limit_old))
res_all = pd.DataFrame(rows)
res_all['announced share'] = res_all['announced'] / res_all['blockages']
res_all[['ci low', 'ci high']] = [wilson(a, n) for a, n in zip(res_all['announced'], res_all['blockages'])]
display(res_all.round(3).set_index(['printer', 'model']))
print(res_all.groupby('model')[['AUC', 'recall', 'precision', 'false episodes / week']].mean().round(3))

**Does the frozen model transfer? Compare with your expectation. Which of the two models holds up better, and on which metric?**

*Your answer:*

## Step 3: Has the normal level moved?
A frozen model assumes that "normal" has not changed. Compare the energy per minute of the 2026 printers with the reference of 2024/25.

In [ ]:
drift = pd.DataFrame({k: {'mean E120': d['E120'].mean(), 'sd E120': d['E120'].std()} for k, (d, _) in printers.items()}).T
drift.loc['2024/25 reference'] = [mu_old, sd_old]
display(drift.round(4))

fig, ax = plt.subplots(figsize=(9, 3.6))
for k, (d, _) in printers.items():
    normal = d[d['y'] == 0]['E120'].resample('MS').mean()
    ax.plot(normal.index, normal.values, marker='o', label=f'printer {k}')
ax.axhline(mu_old, color='k', ls='--', label='reference 2024/25')
ax.set_ylabel('mean E120 of normal operation (Wh/min)'); ax.legend(ncol=4, fontsize=8); plt.show()

**Is there drift? What would you do if the 2026 level were 3 % higher than the reference?**

*Your answer:*

## Step 4: The per-printer baseline
Learn "normal" for every printer anew from its **own** first 70 % of 2026 and test on the last 30 %. All four models are scored on the same test period, so that they are comparable.

In [ ]:
rows = []
for k, (d, blk) in printers.items():
    cut = int(len(d) * 0.7)
    ref, test = d.iloc[:cut], d.iloc[cut:]
    mu, sd = ref['E120'].mean(), ref['E120'].std()
    z_ref, z_test = (ref['E120'] - mu) / sd, (test['E120'] - mu) / sd
    limit = np.quantile(z_ref, ___)
    knn_own = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)).fit(ref[FEATURES], ref['y'])
    z_frozen = (test['E120'] - mu_old) / sd_old
    rows += [
        score_row(k, '1 z-value rule, frozen (2024/25)', test, blk, z_frozen, z_frozen > z_limit_old),
        score_row(k, '2 z-value rule, own reference', test, blk, z_test, z_test > limit),
        score_row(k, '3 KNN, frozen (2024/25)', test, blk, knn_frozen.predict_proba(test[FEATURES])[:, 1], pd.Series(knn_frozen.predict(test[FEATURES]) == 1, index=test.index)),
        score_row(k, '4 KNN, own reference', test, blk, knn_own.predict_proba(test[FEATURES])[:, 1], pd.Series(knn_own.predict(test[FEATURES]) == 1, index=test.index)),
    ]
res = pd.DataFrame(rows)
summary = res.groupby('model').agg(AUC=('AUC', 'mean'), recall=('recall', 'mean'), precision=('precision', 'mean'),
                                   announced=('announced', 'sum'), blockages=('blockages', 'sum'), false_per_week=('false episodes / week', 'mean'))
summary['announced share'] = summary['announced'] / summary['blockages']
display(summary.round(3))

**Does a per-printer baseline beat the frozen model? Look at AUC, the share of announced blockages and the false alarms. What does that mean for the monitor?**

*Your answer:*

## Step 5: How certain is that?
Pool the six printers and give the announced share of the frozen z-value rule with its interval. The printers belong to the same farm and the same process, so they are **not** six independent samples.

In [ ]:
tot = res_all[res_all['model'] == 'z-value rule (frozen)'][['announced', 'blockages']].sum()
lo, hi = wilson(tot['announced'], tot['blockages'])
print(f"frozen z-value rule, whole of 2026: {int(tot['announced'])} of {int(tot['blockages'])} blockages announced = {tot['announced'] / tot['blockages']:.0%} (95 % interval {lo:.0%} to {hi:.0%})")
print('false alarm episodes per printer and week:', res_all[res_all['model'] == 'z-value rule (frozen)']['false episodes / week'].round(2).tolist())

# Chance level: shift the alarm series of every printer by a random time (the alarm rate stays the same, the link to the blockages is destroyed)
def announced_fast(alarm_times, blockages):
    t = np.sort(alarm_times.values)
    return sum(np.searchsorted(t, b, 'right') > np.searchsorted(t, b - np.timedelta64(6, 'h'), 'left') for b in blockages.values)

rng = np.random.default_rng(0)
null = []
for _ in range(200):
    n_ann = n_blk = 0
    for k, (d, blk) in printers.items():
        alarm = np.roll(((d['E120'] - mu_old) / sd_old > z_limit_old).values, rng.integers(500, len(d) - 500))
        b = blk[(blk >= d.index[0]) & (blk <= d.index[-1])]
        n_ann += announced_fast(d.index[alarm], b); n_blk += len(b)
    null.append(n_ann / n_blk)
print(f'chance level with shifted alarms: {np.mean(null):.0%} (5 to 95 %: {np.quantile(null, 0.05):.0%} to {np.quantile(null, 0.95):.0%}) against {tot["announced"] / tot["blockages"]:.0%} observed')

**What can you claim about the transfer, and what not? Which threshold would you give the monitor, and who decides?**

*Your answer:*